# ACFR extraction with Google Drive in Colab

This optional notebook runs the standalone program in your own Google Colab session.
It has not been run against your Google account during development. Sign in with the
account that can access ACFRS. Mounting Drive asks for permissions; no credentials
are embedded here. Original PDFs are read only. New source, results and release
copies are saved to a new timestamped project location under your chosen Drive
folder. Review these cells before running. Colab sessions are temporary; checkpoint
files remain local until the final save cell completes.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


Upload the downloaded **acfr-caption-extractor-0.3.0.zip** release (not the input PDF ZIP).
The source folder must contain acfr_extractor.py. This upload is a temporary copy.


In [ ]:
from pathlib import Path
import zipfile, tempfile, subprocess, sys, shutil, datetime
uploaded = files.upload()
zips = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(zips) == 1, 'Upload exactly one release ZIP'
release_zip = zips[0]
work = Path(tempfile.mkdtemp(prefix='acfr-'))
with zipfile.ZipFile(release_zip) as archive:
    for info in archive.infolist():
        target = (work / info.filename).resolve()
        assert target.is_relative_to(work.resolve()), 'Unsafe archive member'
    archive.extractall(work)
sources = list(work.rglob('acfr_extractor.py'))
assert len(sources) == 1, 'Expected one application in the release'
source = sources[0].parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(source / 'requirements.txt')], check=True)
if not shutil.which('tesseract'):
    subprocess.run(['apt-get', 'update'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr', 'tesseract-ocr-eng', 'poppler-utils', 'git'], check=True)
subprocess.run(['tesseract', '--list-langs'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=source, check=True)


Enter the mounted **ACFRS input folder path**, for example `/content/drive/MyDrive/ACFRS`.
This is a path, not the folder's web URL. If the folder is shared with you, add a
shortcut to My Drive first if necessary. Verify the inventory before proceeding;
this cell does not assume the original web-folder ID maps to a particular path.


In [ ]:
input_folder = Path(input('Mounted Drive ACFRS folder path: ').strip()).resolve()
assert input_folder.is_relative_to(Path('/content/drive')), 'Choose a mounted Drive folder'
assert input_folder.is_dir(), 'Folder does not exist'
project = input_folder / 'ACFR Extraction Software'
inputs = sorted(p for p in input_folder.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf' and not p.is_relative_to(project))
print('PDF count:', len(inputs))
for p in inputs[:30]: print(p.relative_to(input_folder))
assert inputs, 'No PDF inputs found'


Run extraction. This reads PDFs from mounted Drive and uses a temporary local
results/checkpoint directory. No original PDFs are modified. The processing log
reports problems; OCR runs locally. If you have source Drive IDs, supply them in a
metadata JSON as described in the manual; mounted filesystem paths alone do not
reveal those IDs. Large collections can exceed Colab runtime limits.


In [ ]:
local_results = work / 'results'
command = [sys.executable, str(source / 'acfr_extractor.py'), '--input', str(input_folder), '--output', str(local_results), '--ocr', 'auto', '--workers', '2', '--resume']
result = subprocess.run(command, cwd=source)
print('Exit status:', result.returncode)
print('0 = no document failures; 1 = some PDFs failed; review CSV warnings in either case.')


Save permanent deliverables into Drive. New timestamped directories prevent
existing results from being overwritten. This cell copies source, the release ZIP,
and completed CSVs; it never moves or renames input PDFs. Validation metrics are
not measured automatically unless a matching verified reference is provided.


In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
for name in ['Source Code', 'Development Results', 'Validation Results', 'Releases']:
    (project / name).mkdir(parents=True, exist_ok=True)
source_destination = project / 'Source Code' / stamp
shutil.copytree(source, source_destination, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
results_destination = project / 'Development Results' / stamp
results_destination.mkdir()
for p in local_results.glob('*.csv'): shutil.copy2(p, results_destination / p.name)
validation = local_results / 'validation_report.csv'
if validation.exists(): shutil.copy2(validation, project / 'Validation Results' / (stamp + '_validation_report.csv'))
shutil.copy2(release_zip, project / 'Releases' / (stamp + '_' + release_zip.name))
# SQLite is closed now; a copy is safe. Copy it back locally before resuming later.
checkpoint = local_results / 'progress.sqlite'
if checkpoint.exists(): shutil.copy2(checkpoint, results_destination / checkpoint.name)
print('Saved to:', project)
print('Source:', source_destination)
print('Results:', results_destination)
